# Follow research as it works

`wait()` prints the research's log while it waits: a line for each finished step, and the step in progress as the last line, with a running timer. These are the same lines the API Playground shows and the `epoch` CLI prints. In a notebook they appear in the cell's output; in a terminal they go to stderr, so a script's own output stays clean.

Set `EPOCH_API_KEY` before starting the kernel, and `EPOCH_API_URL` for an environment other than production. Research takes a minute or two.

In [ ]:
from epoch_sdk import Epoch

epoch = Epoch()  # Reads EPOCH_API_KEY, and EPOCH_API_URL when set.

## Ask

The outputs say what comes back: a written summary and a table with two columns.

In [ ]:
research = epoch.research.create(
    task="Compare MSFT and SPY volatility in 2025.",
    context=["Use daily returns in USD.", "Annualize volatility using 252 trading days."],
    outputs=[
        {"name": "summary", "type": "text", "instructions": "Explain the comparison in three sentences."},
        {"name": "volatility", "type": "table", "schema": [
            {"name": "symbol", "type": "string"},
            {"name": "annualized_volatility", "type": "float64"},
        ]},
    ],
)
research = epoch.research.wait(research.id)

## Read the answer

If Epoch asked a question instead, the log's last line says what it asked; answer it with `epoch.research.answer` and wait again.

In [ ]:
if research.status == "completed":
    ref = research.result.response
    answer = epoch.responses.get(ref.id, version=ref.version)
    for part in answer.content.outputs:
        if part.type == "text":
            print(part.value)
        elif part.type == "table":
            print(part.name, epoch.responses.rows(ref.id, part.name, version=ref.version))
else:
    print(research.status, research.questions or research.error)

## Run it again with new dates

The same study, for another period. A run has no event stream, so `wait()` reads it every 2 seconds while it logs.

In [ ]:
run = epoch.executions.create(research_id=research.id, start="2026-01-01", end="2026-11-15")
run = epoch.executions.wait(run.id)

## Where the log goes

- `wait(..., log=False)` prints nothing.
- `wait(..., log=True)` prints it anywhere, including CI and pipes: finished lines only, without colour.
- `wait(..., log=open("research.log", "w"))` writes it to that file.

Colour is left out when `NO_COLOR` is set.